# C5 · Sustracción de halo SGF

**Spec:** [`docs/spec_C5_codex_sgf_subtraction.md`](../docs/spec_C5_codex_sgf_subtraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Sustrae el halo estelar por diversidad espectral con filtrado Savitzky-Golay (Haffert et al. 2019; Julo et al. 2025 App. A.3) y extrae el producto box3 del residual.

| | |
|---|---|
| **Entrada** | `stage02` stack + posiciones (B3) + PSF (C1, solo apcorr) |
| **Salida (QC/productos)** | `stages/spec_sgf_qc.json`, `spec_sgf_object.fits`, cubo residual |
| **Consume aguas abajo** | D1 v3, G1, E4; cubo residual → E1b (mapa FoV) |


## Qué hace C5 y por qué existe

C5 implementa el método **estado-del-arte de literatura** (HRSDI/SGF): por spaxel, divide por un espectro estelar de referencia (mediana de spaxels con flujo en 0.01–0.1×Fmax), suaviza el ratio con Savitzky-Golay (d=1, W̆=101 canales) y sustrae referencia×ratio_suavizado.

**Deliberadamente sin enmascarar líneas y sin PCA**: es la línea base cuyos sesgos cuantifica Julo et al. 2025 — auto-sustracción de líneas y continuo negativo vecino, con profundidad exacta (en el modelo de juguete) `C̃_P/L̂_P = −(R/(1−R))·(C_S/L_S)` (Ec. 1). El QC registra ese **predictor por línea** con C_S/L_S medido en la referencia del run, y la corrección práctica es el throughput E4/E3 (como Jorquera et al. 2024) — nunca un parche dentro de C5.

Los oráculos analíticos del paper están verificados en `tests/test_halosub_toy.py` (Fig. 2d y Ec. 1 a 8 decimales) y el contrato de la etapa en `tests/test_halosub_stages.py`.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_x04_sgf --run-id $RUN
```

Ligero (~1 min: un savgol por exposición + extracción).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_sgf_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_x04_sgf --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_sgf_qc.json', RUN_ID)
nb.show(qc, keys=['halosub.n_exposures', 'sgf.window', 'errors.mode', 'checks.v1_reference_ok', 'checks.v2_far_continuum_ok', 'checks.v4_scale_convention_ok'], title='C5')


## Los chequeos del QC, en físico

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v1_reference_ok` | **¿Hay bastante halo para construir la referencia estelar?** ≥ 50 spaxels conservados por exposición. | La referencia es ruido: el método resta ruido en vez de halo. |
| `v2_far_continuum_ok` | **¿La sustracción deja el fondo en cero donde no hay nada?** Mediana del residuo en los controles, en continuo lejos de líneas, compatible con 0. | Hay un pedestal residual: todo lo extraído después lleva ese sesgo. |
| `v3_predictor_written` | **¿Cuánta señal de línea se espera perder?** El SGF filtra el continuo por construcción y muerde también la línea; el predictor (Ec. 1 de Julo+25) queda registrado para cada línea estándar en cobertura. | No hay con qué interpretar el flujo de línea de este método. |
| `v4_scale_convention_ok` | **¿Queda trazable en qué escala está el producto?** Cabeceras `BKGMODE`/`SCALEREF` y espectros de control persistidos. | D1 no puede comparar este método con los otros sin adivinar la convención. |
| `v5_no_pca` | **¿Se coló PCA?** Declara explícitamente `pca_applied = false`. | Sería saltarse una decisión congelada (PCA descartado en D1). |


## Evidencia: predictor de auto-sustracción y continuo negativo

Predictor Ec. 1 por línea de ciencia (con C_S/L_S medido en la referencia) y fracción de canales < −2σ en las bandas laterales del producto.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C5', 'stages/spec_sgf_qc.json'):
        q = nb.load_qc('stages/spec_sgf_qc.json', RUN_ID)
        print(f"exposiciones: {q['halosub']['n_exposures']}  spaxels ref: {q['halosub']['n_spaxels_kept']}")
        for row in q['self_subtraction_predictor']:
            if row['in_range']:
                print(f"  {row['line']:8s} C_S/L_S={row['cs_over_ls']:.3f}  R={row['R']:.4f}  "
                      f"predictor C̃_P/L̂_P={row['predictor']:+.4f}")
        for row in q['negative_continuum']:
            if row['frac_below_minus2sigma'] is not None:
                print(f"  {row['line']:8s} frac(<−2σ) en bandas laterales = {row['frac_below_minus2sigma']:.3f}")
        print('checks:', {k: v for k, v in q['checks'].items()})


## Plot — residual SGF y espectro del compañero

Colapso del cubo residual (7000–8500 Å) y espectro `spec_sgf_object` con la banda ±1σ de controles. Comparar con C4: aquí el halo se remueve por diversidad espectral, no por modelo espacial.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/spec_sgf_qc.json', RUN_ID)
    loc = nb.load_qc('stages/stage01c_qc.json', RUN_ID)
    (cy, cx) = loc['companion']['pos_yx']
    with fits.open(q['products']['residual_cube']) as h:
        res = np.asarray(h['RESIDUAL'].data, float); wave = np.asarray(h['WAVELENGTH'].data, float)
    sel = (wave >= 7000) & (wave <= 8500)
    img = np.nanmedian(res[sel], axis=0)
    hp = fits.open(rd / 'stages' / 'spec_sgf_object.fits')
    w = np.asarray(hp[1].data['wave_A'], float); f = np.asarray(hp[1].data['flux'], float); hp.close()
    C = np.load(rd / 'stages' / 'spec_sgf_controls.npz')['control_spectra']
    sig = np.nanstd(C, axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), width_ratios=[1, 2])
    v = np.nanpercentile(img, [5, 99])
    axes[0].imshow(img, origin='lower', cmap='magma', vmin=v[0], vmax=v[1])
    axes[0].plot(cx, cy, 'o', mfc='none', mec='lime', ms=12); axes[0].axis('off')
    axes[0].set_title('residual SGF (mediana 7000–8500 Å)')
    axes[1].fill_between(w, -sig, sig, color='0.85', label='±1σ controles')
    axes[1].plot(w, f, lw=0.4, color='tab:blue', label='compañero (sgf)')
    axes[1].axvline(6563, color='tab:red', ls=':', label='Hα'); axes[1].axhline(0, color='0.6', lw=0.6)
    axes[1].set_ylim(np.nanpercentile(f, 2), np.nanpercentile(f, 98))
    axes[1].set_xlabel('λ [Å]'); axes[1].legend(fontsize=8)
    axes[1].set_title('C5 · espectro sgf del compañero')
    fig.tight_layout()
    outdir = rd / 'plots' / 'c5_sgf'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'residual_and_spectrum.png', dpi=110); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'sgf'
    PRODUCT_P = 'spec_sgf_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_sgf_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · sgf (C5)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c5_sgf'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Fiel a la literatura** (SavGol d=1, W̆=101, sin máscara de líneas, sin PCA): línea base cuyos sesgos se miden, no se ocultan. · [`docs/spec_C5_codex_sgf_subtraction.md`](../docs/spec_C5_codex_sgf_subtraction.md)
- Predictor Ec. 1 por línea en el QC; la corrección práctica de la auto-sustracción es el throughput E4/E3. · [`docs/plan_integracion_halosub_julo2025.md`](../docs/plan_integracion_halosub_julo2025.md)
- Sustracción por exposición; residuos combinados con el mismo combinador que el cubo madre (comparabilidad D1).


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/spec_sgf_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
    assert q['pca_applied'] is False
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (checkpoint de la spec C5: el usuario aprueba la spec antes de correr). Kernel y contrato verificados con tests sintéticos (2026-07-14).
